# Compare which pairs each method found

Reads the per-pair outcomes for one source and target, every finished run's verdict on every truth pair, and shows which pairs each chosen run found: the counts per combination of runs, an UpSet plot of them, which pairs a name-based method could find at all, each run's precision, recall, reduction ratio and F2 on the level, how many candidates each pair's source was given, how alike the runs are, and the named pairs behind any one combination.

A strategy comparison reports each run's recall on a population, which says how many pairs a run found and not which. Two methods with the same recall on `never` may be finding the same pairs or entirely different ones, and only the second is a reason to combine them.

Each section opens with a cell of settings. Change one and re-run from that cell down. Every reshape is a call into `analysis.pair_outcomes`; the notebook chooses what to look at and displays it.

**Precondition:** `scripts/report_strategy_comparison.py` has written `pair_outcomes.parquet` and `pair_outcome_runs.parquet` and the candidate tables for the source and target. It reads the finished runs on disk and runs nothing. This notebook builds neither. If a source and target are missing, or a run has finished since, build them with:

```bash
uv run python scripts/report_strategy_comparison.py --source gleif --target ie
```

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
if str(ROOT / "scripts") not in sys.path:
    sys.path.insert(0, str(ROOT / "scripts"))
import _bootstrap  # noqa: F401

from workspace.roots import default_workspace_roots

ROOTS = default_workspace_roots(ROOT)

import matplotlib.pyplot as plt
import polars as pl
from IPython.display import HTML, display

from analysis.pair_outcomes import (
    FOUND_BY,
    FOUND_BY_NONE,
    FOUND_BY_SEPARATOR,
    candidates_at,
    cluster_sizes,
    count_by_reach,
    count_found_by,
    cutoffs_for_candidates,
    draw_cluster_sizes,
    draw_found_by_upset,
    draw_name_similarity,
    draw_run_overlap,
    drop_unanimous_pairs,
    latest_runs,
    level_metrics,
    pair_found_matrix,
    pairs_found_by,
    random_pair_level,
    run_columns,
    run_overlap,
    select_runs,
    with_name_similarity,
)
from blocking.run_layout import ComparisonArtefact, iter_blocking_comparisons

# analysis.pair_outcomes forces the Agg backend at import time; switch to the
# notebook backend after importing it so figures render inline.
%matplotlib inline

RUN_COLUMNS = [
    "label",
    "tokenizer",
    "similarity_backend",
    "accelerator_settings",
    "name_transform",
    "top_k",
    "min_similarity",
    "truth_pairs",
    "finished_at",
]

# A table taller than the box scrolls inside it, so its sideways scrollbar
# stays in view; the header row stays put while it does. A value wider than a
# column's cap wraps inside its cell rather than widening the column.
_SCROLLING_TABLE_STYLE = """<style>
.scrolling-table { overflow: auto; max-width: 100%; max-height: 32em; }
.scrolling-table td, .scrolling-table th {
  white-space: normal !important;
  max-width: 30ch;
  overflow-wrap: anywhere;
  vertical-align: top;
}
.scrolling-table thead {
  position: sticky;
  top: 0;
  background: var(--vscode-editor-background, white);
}
</style>"""


def show(frame: pl.DataFrame) -> None:
    """Display every row and column of `frame` as an HTML table in a scrolling box.

    Handing the output only HTML stops an editor falling back to the text table,
    which wraps each cell to fit a fixed width and cannot scroll.
    """
    with pl.Config(tbl_rows=-1, tbl_cols=-1, fmt_str_lengths=200):
        table = frame._repr_html_()
    display(HTML(f'{_SCROLLING_TABLE_STYLE}<div class="scrolling-table">{table}</div>'))


SOURCES_AND_TARGETS = {
    (location.pairing.source_segment, location.pairing.target_system): location
    for location in iter_blocking_comparisons(ROOTS)
    if location.path(ComparisonArtefact.PAIR_OUTCOMES).is_file()
}
print(f"ROOT={ROOT}")
print("Sources and targets with per-pair outcomes:")
show(
    pl.DataFrame(
        list(SOURCES_AND_TARGETS), schema=["source", "target"], orient="row"
    ).sort("source", "target")
)

## 1) The source, the target and the level

`SOURCE` and `TARGET` are one of the combinations the setup cell listed. `LEVEL` is the name-equality level: `"never"` (cleansed names still differ), `"preprocessed"`, `"cleansed"`, `"basic"`, `"raw"`, `"unknown"` (a name missing), or `None` for every truth pair. `COUNTRY` is one country code, or `None` for every country.

The table is every finished run for that source and target, with the settings each was made under, newest first.

In [60]:
SOURCE = "gleif"
TARGET = "ie"
LEVEL = "never"
COUNTRY = None

In [61]:
location = SOURCES_AND_TARGETS[(SOURCE, TARGET)]
all_runs = pl.read_parquet(location.path(ComparisonArtefact.PAIR_OUTCOME_RUNS))
outcomes = pl.read_parquet(location.path(ComparisonArtefact.PAIR_OUTCOMES))
candidates_path = location.path(ComparisonArtefact.PAIR_OUTCOME_CANDIDATES)
candidates = (
    pl.read_parquet(candidates_path)
    if candidates_path.is_file()
    else pl.DataFrame(
        schema={"label": pl.Utf8, "min_similarity": pl.Float64, "rank": pl.Int64, "candidates": pl.Int64}
    )
)
source_candidates = pl.read_parquet(
    location.path(ComparisonArtefact.PAIR_OUTCOME_SOURCE_CANDIDATES)
)
print(f"{all_runs.height} run(s), {outcomes.height:,} pair outcome(s): {location.directory}")
print("Countries:", ", ".join(sorted(outcomes["country"].unique().to_list())))
show(all_runs.sort("finished_at", descending=True).select(RUN_COLUMNS))

21 run(s), 401,205 pair outcome(s): C:\Devel\Brunel\blocking\artifacts\blocking\data\ie\data\gleif\comparison
Countries: ie


label,tokenizer,similarity_backend,accelerator_settings,name_transform,top_k,min_similarity,truth_pairs,finished_at
str,str,str,str,str,i64,f64,i64,str
"""sbert/b6c9d6c4ef3a""","""wordpiece:promoted""","""dense_brute""","""exact_name_filter=true""","""cleanse:default;preprocess:default""",20,0.75,19105,"""2026-09-21T08:13:43.316986+00:00"""
"""sentencepiece/ec58527f270c""","""sentencepiece:naive""","""kmeans""","""exact_name_filter=true;kmeans_init=""sample"";n_clusters=300;seed=42""","""cleanse:default;preprocess:default""",20,0.75,19105,"""2026-09-21T00:03:14.477625+00:00"""
"""wordpiece/fad36375762f""","""wordpiece:naive""","""kmeans""","""exact_name_filter=true;kmeans_init=""sample"";n_clusters=300;seed=42""","""cleanse:default;preprocess:default""",20,0.75,19105,"""2026-09-21T00:01:05.681598+00:00"""
"""sentencepiece/418502f5dbb7""","""sentencepiece:promoted""","""kmeans""","""exact_name_filter=true;kmeans_init=""sample"";n_clusters=300;seed=42""","""cleanse:default;preprocess:default""",20,0.75,19105,"""2026-09-20T16:57:01.242538+00:00"""
"""sentencepiece/d8630f7871c8""","""sentencepiece:naive""","""kmeans""","""exact_name_filter=true;kmeans_init=""sample"";n_clusters=300;seed=42""","""cleanse:default;preprocess:default""",20,0.75,19105,"""2026-09-20T16:54:40.952207+00:00"""
"""wordpiece/e40bd3182e7c""","""wordpiece:naive""","""kmeans""","""exact_name_filter=true;kmeans_init=""sample"";n_clusters=300;seed=42""","""cleanse:default;preprocess:default""",20,0.75,19105,"""2026-09-20T16:49:23.995872+00:00"""
"""tfidf/dcbb9a02a255""","""wordpiece:promoted""","""kmeans""","""exact_name_filter=true;kmeans_init=""smart"";n_clusters=900;seed=42""","""cleanse:default;preprocess:default""",20,0.75,19105,"""2026-09-19T20:59:20.337016+00:00"""
"""tfidf/aa02fa4992b2""","""wordpiece:promoted""","""kmeans""","""exact_name_filter=true;kmeans_init=""sample"";n_clusters=2400;seed=42""","""cleanse:default;preprocess:default""",20,0.75,19105,"""2026-09-19T20:23:58.418792+00:00"""
"""tfidf/444fdad44e8e""","""wordpiece:promoted""","""kmeans""","""exact_name_filter=true;kmeans_init=""sample"";n_clusters=900;seed=42""","""cleanse:default;preprocess:default""",20,0.75,19105,"""2026-09-19T19:41:42.040246+00:00"""


## 2) The runs compared

Each filter is a list of values, or `None` to admit every run. `LATEST_PER` then keeps the most recently finished run per value of one column (`"representation"`, `"tokenizer"`, `"similarity_backend"`), or `None` to keep every run left. `LABELS`, a list of run labels, picks exact runs and overrides all of that.

A run's column below is named by its `LATEST_PER` value where one was used, else by its label.

Two runs are only a clean comparison where they differ in the one thing being compared. The table shows each chosen run's settings so that is visible; nothing here enforces it.

In [62]:
REPRESENTATION = None
TOKENIZER = None
SIMILARITY_BACKEND = None
LATEST_PER = "representation"
LABELS = None

In [63]:
chosen = select_runs(
    all_runs,
    labels=LABELS,
    representation=REPRESENTATION,
    tokenizer=TOKENIZER,
    similarity_backend=SIMILARITY_BACKEND,
)
run_names = None
if not LABELS and LATEST_PER is not None:
    chosen = latest_runs(chosen, by=LATEST_PER)
    run_names = dict(zip(chosen["label"], chosen[LATEST_PER]))
print(f"{chosen.height} run(s) compared")
if chosen.select("population_key", "truth_key").unique().height > 1:
    print("WARNING: the chosen runs did not all score one population against one truth")
show(chosen.select(RUN_COLUMNS))

4 run(s) compared


label,tokenizer,similarity_backend,accelerator_settings,name_transform,top_k,min_similarity,truth_pairs,finished_at
str,str,str,str,str,i64,f64,i64,str
"""sbert/b6c9d6c4ef3a""","""wordpiece:promoted""","""dense_brute""","""exact_name_filter=true""","""cleanse:default;preprocess:default""",20,0.75,19105,"""2026-09-21T08:13:43.316986+00:00"""
"""sentencepiece/ec58527f270c""","""sentencepiece:naive""","""kmeans""","""exact_name_filter=true;kmeans_init=""sample"";n_clusters=300;seed=42""","""cleanse:default;preprocess:default""",20,0.75,19105,"""2026-09-21T00:03:14.477625+00:00"""
"""tfidf/dcbb9a02a255""","""wordpiece:promoted""","""kmeans""","""exact_name_filter=true;kmeans_init=""smart"";n_clusters=900;seed=42""","""cleanse:default;preprocess:default""",20,0.75,19105,"""2026-09-19T20:59:20.337016+00:00"""
"""wordpiece/fad36375762f""","""wordpiece:naive""","""kmeans""","""exact_name_filter=true;kmeans_init=""sample"";n_clusters=300;seed=42""","""cleanse:default;preprocess:default""",20,0.75,19105,"""2026-09-21T00:01:05.681598+00:00"""


## 3) Which pairs each run found

`pair_found_matrix` gives one row per truth pair at the chosen level with a found-or-missed column per run. A pair is in when any chosen run placed it at that level, and carries every chosen run's verdict.

Runs made at one threshold do not spend the same comparisons, since cosine is on a different scale per representation, so a run that keeps more candidates finds more pairs for that reason alone. `CUTOFFS` maps a run label to a threshold above the one it was made at: the run is read as if made there, a pair counting as found only where its similarity reaches the cutoff. `BUDGET`, when above 0, sets every run's cutoff to the lowest that holds it within that many candidate pairs, and `CUTOFFS` then overrides it per run. `CAP_PER_SOURCE`, when above 0, holds every run to that many candidates per source, by rank, which bounds cost the same way on every corpus where a cutoff does not. A run can only be read more tightly than it was made, since pairs below its own threshold or beyond its `top_k` were never kept.

The first table is each run's own total, with its cutoff and the candidates it keeps there. The second counts the pairs by the combination of runs that found them: `tfidf` alone is the pairs only tfidf found, `(none)` the pairs no chosen run found. The pairs no run found are usually most of a `never` population and flatten every other bar, so `DROP_NONE` leaves them out of the plot.

`COUNTS` sets what each bar of the plot counts: `"exact"` the pairs found by exactly those runs, as the table does; `"at_least"` the pairs found by at least those runs, so a pair sbert, tfidf and wordpiece all found also counts under `sbert + tfidf`, which gives each combination everything it recovers; `"both"` draws the at-least bars above the axis and the exact bars, dimmed, below it.

`EXCLUDE_BELOW_RANDOM`, off by default, sets aside the pairs whose names are no more alike than two unrelated names (section 4 explains the level, at `RANDOM_QUANTILE`): no name-based method could find them. With it on, everything from here down, the counts, the plot, the metrics and the overlap, is read over the pairs such a method could find.

In [ ]:
EXCLUDE_BELOW_RANDOM = False
RANDOM_QUANTILE = 0.99
CUTOFFS = {}
BUDGET = 0
CAP_PER_SOURCE = 0
DROP_NONE = True
DROP_ALL = False
COUNTS = "both"

In [ ]:
column_of = lambda label: (run_names or {}).get(label, label)
made_at = dict(chosen.select("label", "min_similarity").iter_rows())
max_rank = CAP_PER_SOURCE or None
wanted = (
    cutoffs_for_candidates(candidates, chosen, budget=BUDGET, max_rank=max_rank)
    if BUDGET > 0
    else {}
)
wanted.update(CUTOFFS)
cutoffs = {label: max(wanted.get(label, made), made) for label, made in made_at.items()}

matrix = pair_found_matrix(
    outcomes,
    chosen,
    name_equality=LEVEL,
    country=COUNTRY,
    run_names=run_names,
    min_similarity={label: cut for label, cut in cutoffs.items() if cut > made_at[label]},
    max_rank=max_rank,
)
random_level = random_pair_level(matrix, quantile=RANDOM_QUANTILE)
marked = with_name_similarity(matrix, random_level=random_level)
if EXCLUDE_BELOW_RANDOM:
    matrix = marked.filter(pl.col("above_random")).drop("name_similarity", "above_random")
    print(f"pairs no more alike by name than random (<= {random_level:.3f}) set aside")
remainder = drop_unanimous_pairs(matrix, found_by_none=DROP_NONE, found_by_all=DROP_ALL)
found_by_any = drop_unanimous_pairs(matrix).height
total = max(matrix.height, 1)
population = "residual pairs" if LEVEL == "never" else f"level {LEVEL!r} pairs"
print(
    f"{matrix.height:,} pair(s) at level {LEVEL!r}; {found_by_any:,} found by at least "
    f"one run ({found_by_any / total:.1%}); {remainder.height:,} plotted"
)

kept = candidates_at(candidates, cutoffs, max_rank=max_rank)
labels = list(made_at)
show(
    pl.DataFrame(
        {
            "run": [column_of(label) for label in labels],
            "cutoff": [cutoffs[label] for label in labels],
            "candidates kept": [kept.get(label) for label in labels],
            "found": [int(matrix[column_of(label)].fill_null(False).sum()) for label in labels],
        }
    ).with_columns((pl.col("found") / total).alias("recall"))
)
show(count_found_by(matrix))

fig = plt.figure(figsize=(11, 5.5))
draw_found_by_upset(
    fig, remainder, title=f"{SOURCE} -> {TARGET}: {population} by source", counts=COUNTS
)
plt.show()

## 4) Which pairs a name-based method could find

`random_pair_level` scores each source name against another pair's target name over seeded shuffles, and the `RANDOM_QUANTILE` of those scores, set in section 3, is the name similarity two unrelated names reach. A pair whose own names score no higher is, by name, no different from a non-match, so no name-based method can be held to it.

This section always reads every pair of the level, whatever `EXCLUDE_BELOW_RANDOM` says. The table counts the pairs by whether any chosen run found them and whether they are above that level: the pairs no run found split into those a method could have found, the real gap, and those it could not. The histogram shows the same split by similarity, the dashed line at the random level.

In [ ]:
print(f"random-pair level at the {RANDOM_QUANTILE:.0%} quantile: {random_level:.3f}")
show(count_by_reach(marked))

fig, ax = plt.subplots(figsize=(10, 4.5))
draw_name_similarity(
    ax, marked, random_level=random_level, title=f"{SOURCE} -> {TARGET}: {population} by name similarity"
)
plt.show()

## 5) Each run's precision, recall, reduction ratio and F2 on the level

Read at the same cutoffs and cap as section 3, and on the level's own sources: recall is the level's pairs found over its pairs; the candidates are what each run proposed for those sources; precision is the truth pairs found for those sources, at any level, over those candidates; the reduction ratio is one less those candidates over the sources times the target size; F2 weights recall four times precision. `recall_above_random` is the recall over the pairs above the random level of section 4.

On `never` these are the figures a method is judged by: a pair equal by name at any level is found by an exact join before any method runs. Precision is a floor, since a candidate outside the truth set may be an unlabelled match.

In [67]:
show(
    level_metrics(
        outcomes,
        chosen,
        source_candidates,
        name_equality=LEVEL,
        country=COUNTRY,
        run_names=run_names,
        min_similarity={label: cut for label, cut in cutoffs.items() if cut > made_at[label]},
        max_rank=max_rank,
        random_level=random_level,
        above_random_only=EXCLUDE_BELOW_RANDOM,
    )
)

run,sources,pairs,found,recall,candidates,truth_found,precision,reduction_ratio,f2,recall_above_random
str,i64,i64,i64,f64,i64,i64,f64,f64,f64,f64
"""sbert""",775,775,232,0.299355,3215,232,0.072162,0.999995,0.18369,0.57037
"""sentencepiece""",775,775,78,0.100645,466,78,0.167382,0.999999,0.109366,0.190123
"""tfidf""",775,775,149,0.192258,1055,149,0.141232,0.999998,0.179302,0.367901
"""wordpiece""",775,775,94,0.12129,484,94,0.194215,0.999999,0.131138,0.22963


## 6) How many candidates each pair's source was given

Each pair takes the number of candidates the run gave its source, at the same cutoffs and cap: the cluster the right target had to be in. A pair in the `0` group was given nothing, a miss that cost the run nothing, and where most of a run's misses sit there, its threshold, not its ranking, is what lost them. Each bar is stacked `found`; `findable`, missed but above the random level of section 4; and `random`, missed and at or below it, which no name-based method could be held to.

In [ ]:
sizes = cluster_sizes(
    matrix,
    source_candidates,
    chosen,
    run_names=run_names,
    min_similarity={label: cut for label, cut in cutoffs.items() if cut > made_at[label]},
    max_rank=max_rank,
    random_level=random_level,
)
fig, ax = plt.subplots(figsize=(12, 5))
draw_cluster_sizes(
    ax, sizes, title=f"{SOURCE} -> {TARGET}: {population} by source cluster size"
)
plt.show()
show(sizes.pivot(on=["run", "outcome"], index="cluster_size", values="pairs"))

## 7) How alike the runs are

Every run against every run on the pairs of section 3. Jaccard is the pairs both found over the pairs either found. Containment, read along a row, is the share of that run's pairs the other run found too: a row near one says the run adds nothing the other lacks, which Jaccard hides when the two find different numbers of pairs.

In [ ]:
overlap = run_overlap(matrix)
fig, (left, right) = plt.subplots(1, 2, figsize=(13, 5.5))
draw_run_overlap(left, overlap, value="jaccard", title="Jaccard")
draw_run_overlap(right, overlap, value="containment", title="containment")
fig.suptitle(f"{SOURCE} -> {TARGET}: {population} by source")
fig.tight_layout()
plt.show()

## 8) The pairs behind one combination

The first cell lists each combination of runs with its pair count, numbered by `found_by_id`. Set `FOUND_BY_ID` to the number of the row to look at; `0` is `(none)` when it heads the table. The table is the pairs exactly those runs found and no other chosen run did, with the names as registered and as cleansed.

In [70]:
combinations = count_found_by(matrix).with_row_index("found_by_id")
show(combinations)

found_by_id,found_by,pairs,share
u32,str,u32,f64
0,"""(none)""",519,0.669677
1,"""sbert""",67,0.086452
2,"""sbert + tfidf""",66,0.085161
3,"""sbert + sentencepiece + tfidf + wordpiece""",35,0.045161
4,"""sbert + tfidf + wordpiece""",27,0.034839
5,"""sbert + sentencepiece + wordpiece""",15,0.019355
6,"""sbert + sentencepiece + tfidf""",10,0.012903
7,"""tfidf""",10,0.012903
8,"""sentencepiece + wordpiece""",9,0.011613


In [71]:
FOUND_BY_ID = 1

In [72]:
combination = combinations.filter(pl.col("found_by_id") == FOUND_BY_ID)[FOUND_BY].item()
found_by = [] if combination == FOUND_BY_NONE else combination.split(FOUND_BY_SEPARATOR)
behind = pairs_found_by(matrix, found_by)
print(f"{behind.height:,} pair(s) found by exactly {combination}")
names = [
    column
    for column in ("source_name", "source_name_cleansed", "target_name", "target_name_cleansed")
    if column in behind.columns
]
show(behind.select("country", *names, "source_id", "target_id"))

67 pair(s) found by exactly sbert


country,source_name,source_name_cleansed,target_name,target_name_cleansed,source_id,target_id
str,str,str,str,str,str,str
"""ie""","""POLITRA B.V. SARL""","""politra bv sarl""","""POLITRA B.V. S.A.R.L.""","""politra bvsarl""","""gleif://2138003QNG66PKGJP383""","""ie://907083"""
"""ie""","""DECO 2015-RUBY DESIGNATED ACTIVITY COMPANY""","""deco 2015 ruby dac""","""DECO 2016-RUBY DESIGNATED ACTIVITY COMPANY""","""deco 2016 ruby dac""","""gleif://2138008FLH27WH9U9C81""","""ie://562946"""
"""ie""","""PAUL RYAN INDEPENDENT PENSIONS & FINANCIAL CONSULTANTS LIMITED""","""paul ryan independent pensions & financial consultants ltd""","""PAUL RYAN FINANCIAL ADVISORS LIMITED""","""paul ryan financial advisors ltd""","""gleif://2138009MFEBOV2HTEY79""","""ie://265790"""
"""ie""","""TIROL INVESTMENTS DESIGNATED ACTIVITIY COMPANY""","""tirol investments designated activitiy co""","""TIROL INVESTMENTS DESIGNATED ACTIVITY COMPANY""","""tirol investments dac""","""gleif://213800IN34YTVRLIJ283""","""ie://664380"""
"""ie""","""MORAN'S PHARMACY LIMITED""","""moran s pharmacy ltd""","""MORANS PHARMACY (CLIFDEN) LIMITED""","""morans pharmacy clifden ltd""","""gleif://213800KJ153KMOLM5219""","""ie://111098"""
"""ie""","""Blue Insurance Limited""","""blue insurance ltd""","""Cover-More Blue Insurance Services LIMITED""","""cover more blue insurance services ltd""","""gleif://2549008U72X94FMN8T07""","""ie://345681"""
"""ie""","""DIASPORA MANAGING LTD""","""diaspora managing ltd""","""DIASPORA MANAGEMENT LIMITED""","""diaspora management ltd""","""gleif://254900ESF0V68KRH2D48""","""ie://469997"""
"""ie""","""537482 Reg. Name BROADRIVER II (IRELAND) DESIGNATED ACTIVITY COMPANY""","""537482 reg name broadriver ii ireland dac""","""BROADRIVER II (IRELAND) DESIGNATED ACTIVITY COMPANY""","""broadriver ii ireland dac""","""gleif://254900GR98FA5AOESZ93""","""ie://537482"""
"""ie""","""Opel Bank GmbH Niederlassung Ireland""","""opel bank gmbh niederlassung ireland""","""OPEL BANK GMBH""","""opel bank gmbh""","""gleif://529900N37SG6NUGG1F37""","""ie://908520"""


The same pairs as tab-separated text, one per line with a header, to paste into Word and convert with Insert > Table > Convert Text to Table, separated at tabs. `PASTE_COLUMNS` picks the columns.

In [ ]:
PASTE_COLUMNS = ["source_name_cleansed", "target_name_cleansed"]

pasteable = behind.select(PASTE_COLUMNS)
print("\t".join(PASTE_COLUMNS))
print("\n".join("\t".join("" if value is None else str(value) for value in row) for row in pasteable.rows()))